# 22: Flow and functions in Python

Author: Greg Wray  
Date: 2026-MAR-13    


## Example of functions and control flow

The following program converts a protein FASTA file into a more easily computable format. This program illustrates the use of functions and control flow, as well as working with dictionaries and using context managers for read and write file operations.

FASTA files have an awkward organization, with each entry on two consecutive lines; in addition, there may or may not be a description following the file identifier and the sequence may or may not contain newline escape sequences. 

This program reads a protein FASTA file and converts the contents into a Python dictionary with the following organization:      
* key = identifier (with '>' removed)
* value = list composed of two items: the description and the sequence (with any newlines and whitespace removed)

It includes two functions to work with the dictionary: 
* add a field representing the length of the protein
* output the dictionary in `.csv` fomat

In [21]:
### program to read a FASTA file and convert to dictionary 
# adds a field for protein length and ouputs dictionary as a .csv file

# define functions
def add_length(dict):
    """ add a field containing protein length in number of AAs
        input: dictionary defined in main program
        return: same with additional field (modifies in place)
    """
    for k, v in dict.items():
        prot_length = len(v[1])
        dict[k].append(prot_length)
    return dict

def to_csv(dict, fname):
    """ convert the dictionary into a .csv file
        input: dictionary defined in main program
        return: None
        output: writes dictionary contents to file fname.csv 
    """
    with open(fname, 'w') as outfile:
        header = 'ID,description,sequence,length\n'
        outfile.write(header)
        for k, v in dict.items():          # for key:value pairs
            v[0] = v[0].replace(',', ' ')  # remove any commas in description
            v[2] = str(v[2])               # convert integer to string
            next_row = k + ',' + ",".join(v) + '\n'  # construct next row
            outfile.write(next_row)        # write next row to file
    print('wrote data to', fname)

### main program

# read contents of file into a single string
with open('short.fasta', 'r') as infile:
    contents = infile.read()

# create a list of entries
entries = contents.split('>')
entries = entries[1:]                      # get rid of empty first entry

# convert the entries into a dictionary
protein_dict = {}
for i in entries:
    id, remainder = i.split(' ', maxsplit=1)
    desc, seq = remainder.split('\n', maxsplit=1)
    seq = seq.replace('\n', '').strip()
    protein_dict[id] = [desc, seq]

# create a new "column" containing the length of the protein
add_length(protein_dict)

# export the dictionary to a file
to_csv(protein_dict, 'protfile.csv')


wrote data to protfile.csv


In [22]:
# get help for a function
help(to_csv)

Help on function to_csv in module __main__:

to_csv(dict, fname)
    convert the dictionary into a .csv file
    input: dictionary defined in main program
    return: None
    output: writes dictionary contents to file fname.csv



## Functional programming with map() and filter()
`map()` and `filter()` provide a compact and readable way to apply an operation to every item in an iterable by creating **implicit loops**. These functions are useful in situations where you want to apply *simple* functions or filtering operations. Note that the first argument passed to these these functions is a function  and the second argument is an interable.  

In [23]:
# create some lists to work with
list_a = ['armadillo', 'orca', 'three-toed sloth', 'pronghorn', 'aardvark', 'pangolin', 'fruit bat']
list_b = list(range(20))
print(list_a)

['armadillo', 'orca', 'three-toed sloth', 'pronghorn', 'aardvark', 'pangolin', 'fruit bat']


Use `map()` to apply a function to each item in an iterable. `map()` takes two arguments: the function you want to apply and an iterable that you want to apply it to. Note that the function must take exactly one argument, namely the next item in the iterable. Also note that `map()` returns a map object, not a data object of the same type you give it. For this reason, it is common to wrap calls to `map()` with `list()` so that you can work with the result. 

In [94]:
# find the length of each item the traditional way using a loop
my_result = []
for i in list_a:
    my_result.append(len(i))
my_result

[9, 4, 16, 9, 8, 8, 9]

In [95]:
# apply map() to a list of strings
my_result = list(map(len, list_a))
my_result

[9, 4, 16, 9, 8, 8, 9]

In [96]:
# find the square of each item the traditional way using a loop
my_result = []
for i in list_b:
    my_result.append(i**i)
my_result

[1,
 1,
 4,
 27,
 256,
 3125,
 46656,
 823543,
 16777216,
 387420489,
 10000000000,
 285311670611,
 8916100448256,
 302875106592253,
 11112006825558016,
 437893890380859375,
 18446744073709551616,
 827240261886336764177,
 39346408075296537575424,
 1978419655660313589123979]

In [97]:
# apply map() to a list of integers
# we first need to define a function
def square(val):
    return val**val
my_result = list(map(square, list_b))
my_result

[1,
 1,
 4,
 27,
 256,
 3125,
 46656,
 823543,
 16777216,
 387420489,
 10000000000,
 285311670611,
 8916100448256,
 302875106592253,
 11112006825558016,
 437893890380859375,
 18446744073709551616,
 827240261886336764177,
 39346408075296537575424,
 1978419655660313589123979]

Use `filter()` to apply a filter to each item in an iterable, returning only those items that match a condition. Importantly, the condition must be specified in a function. The examples below use simple filters, but the filtering criteria can be as complicated as you want because they are encapsulated in the function. Otherwise, `filter()` works similarly to `map()`.

In [98]:
# filter for strings that start with 'p' the traditional way using a loop
my_result = []
for j in list_a:
    if j.startswith('p'): my_result.append(j)
my_result

['pronghorn', 'pangolin']

In [99]:
# filter for strings that start with 'p' using filter()
# we first need to define a function
def starts_with_p(val):
    return val.startswith('p')
my_result = list(filter(starts_with_p, list_a))
my_result

['pronghorn', 'pangolin']

In [100]:
# filter for numbers divisible by 3 the traditional way using a loop
my_result = []
for k in list_b:
    if k % 3 == 0: my_result.append(k)
my_result

[0, 3, 6, 9, 12, 15, 18]

In [101]:
# filter for numbers divisible by 3 using filter()
# we first need to define a function
def div_by_3(val):
    return val % 3 == 0
my_result = list(filter(div_by_3, list_b))
my_result

[0, 3, 6, 9, 12, 15, 18]

In [66]:
# include an example of using reduce() here, incl requires loading the function

## Lambda functions
Lambda functions are often called "anonymous" or "temporary" functions. The examples above illustrate why they are useful. We can simplify and save several lines of code using `map()` or `filter()`; however, in many cases we need to define a function before we can take adavantage of these functions, which somewhat defeats the purpose. Lambda functions solve this problem by allowing us to define a function directly within the call to  `map()` or `filter()`. Lambda functions are also useful in other situations, including *comprehensions* (below). Lambda functions are commonly used in Python code, so learning how they work is well worth the effort. In general, lambda functions are most useful when (1) you only need to call the function once and (2) the function is simple. 

In [102]:
# apply map() to a list of integers using a lambda function
my_result = list(map(lambda x: x**x, list_b))
my_result

[1,
 1,
 4,
 27,
 256,
 3125,
 46656,
 823543,
 16777216,
 387420489,
 10000000000,
 285311670611,
 8916100448256,
 302875106592253,
 11112006825558016,
 437893890380859375,
 18446744073709551616,
 827240261886336764177,
 39346408075296537575424,
 1978419655660313589123979]

In [103]:
# apply filter() to a list of strings using a lambda function
my_result = list(filter(lambda x: x.startswith('p'), list_a))
my_result

['pronghorn', 'pangolin']

It's also possible to use `map()` and `filter()` with multiple data objects at once.

In [104]:
# find the average of two lists of numbers
my_result = list(map(lambda x, y: (x+y)/2, range(20), range(20,40,2)))
my_result

[10.0, 11.5, 13.0, 14.5, 16.0, 17.5, 19.0, 20.5, 22.0, 23.5]

## Functional programming with comprehensions
**Comprehensions** provide another functional programming construct in Python. Comprehensions are generally more readable than using `map()` and `filter()`, and they are more versatile because you can combine both operations in a single statement. Comprehensions can be applied to any iterator, but are most commonly applied to lists, so you will likely encounter them in the form of **list comprehensions**.

Here are the basic formulas, applied to lists: 
* apply a function or operation to each item: `new_list = [new_item for item in list]` 
* filter items: `new_list = [item for item in list if condition]`
* apply a function or operation to items after filtering: `new_list = [new_item for item in list if condition]`

As with many programming concepts, it's easiest to see how this works by experimenting with some simple cases. The examples below use comprehensions to carry out the for loop, mapping, and filtering operations covered earlier.

In [24]:
# use a comprehension to find the length of each item in a list
[len(x) for x in list_a]

[9, 4, 16, 9, 8, 8, 9]

In [28]:
# use a comprehension to apply a method to each item in a list
[x.title() for x in list_a]

['Armadillo',
 'Orca',
 'Three-Toed Sloth',
 'Pronghorn',
 'Aardvark',
 'Pangolin',
 'Fruit Bat']

In [29]:
# use a comprehension to add 5 to each item in a list
[x + 5 for x in list_b]

[5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]

In [31]:
# use a comprehension to filter for divisible by 3 in a list
[x for x in list_b if x % 3 == 0]

[0, 3, 6, 9, 12, 15, 18]

In [32]:
# use a comprehension to filter for strings longer than 5, then extract the first letter
[x[0].upper() for x in list_a if len(x) > 5]

['A', 'T', 'P', 'A', 'P', 'F']

In the examples above, note that the core of a comprehension is the loop: `for x in list`. To **alter** the items, apply a function or operation on the LHS of the loop. To **filter** the items, add a condition on the RHS of the loop. If you want to do **both**, they can be combined in a single comprehension (as in the final example above). 

Although comprehensions are most commonly applied to lists, they can be used with any iterable type. Note that you need to wrap the comprehension in the appropriate brackets (square, round, curly) to indicate which type of iterable you want back.

In [8]:
# filter for items of length 6 in a set
set_a = {'pink', 'yellow', 'amber', 'indigo', 'gray', 'aqua', 'red', 'green', 'violet'}
{s for s in set_a if len(s) == 6}

{'indigo', 'violet', 'yellow'}

In [33]:
# filter for items of length 6, then extract the first character
{s[0] for s in set_a if len(s) == 6}

{'i', 'v', 'y'}

The examples so far cover the basic syntax and applications of comprehensions. Below are three useful extensions.
    
First, it is possible to specify multiple conditions in the RHS of a comprehension. It is helpful but not required to organize the conditions onto separate lines for readability. The example below applies three conditions to test for specific starting and ending letters of each string in list. The first condition avoids run-time errors that would arise from trying to access a string of length 0. Note that the conditions are separated by whitespace (space or return), not commas.

In [34]:
list_a = ['armadillo', 'orca', 'three-toed sloth', 'pronghorn', 'aardvark', 'pangolin', 'fruit bat']

# filter based on 3 conditions: length, start character, and end character
[r for r in list_a
    if len(r) >7
    if r[0] == 'a'
    if r[-1] == 'k'
]

['armadillo']

Second, it is also possible to use an `if` / `else` structure into the LHS of a comprehension, not to filter, but to perform different operations depending on a condition. A common application is to create a mask for Boolean indexing. (As a quick reminder, we covered Boolean indexing in the first semester with R. This is a fast, versatile way to filter items in a column or other iterable.)

In [86]:
# classify items in a list of string according to criteria
[True if len(t) > 8 else False for t in list_a]

[True, False, True, True, False, False, True]

And third, it is possible to nest loops in a comprehension. This is useful for accessing every item in a matrix so that you can apply a function, carry out an operation, filter, or generate a Boolean index. Nesting can also be useful for simply generating a matrix. The example below generates a 5 x 5 matrix. Note the nesting of square brackets to create an inner and outer loop. If you substitute round brackets in one or the other loop, you can generate a tuple of lists or a list of tuples (or any other compound data structure you wish to create). Also note the use of **anonymous variables**, since we do not need to refer to these variable again (the code will generate exactly the same result if you use i and j or some other variable name).  

In [67]:
# generate a matrix using nested list comprehensions
my_matrix = [[_ for _ in range(5)] for _ in range(5)]
my_matrix

[[0, 1, 2, 3, 4],
 [0, 1, 2, 3, 4],
 [0, 1, 2, 3, 4],
 [0, 1, 2, 3, 4],
 [0, 1, 2, 3, 4]]

## Exception handlers
Errors that arise during the execution of a program are called **exceptions** in Python. Exceptions can become frustrating if you are running programs that take a long time to execute or that run in an unsupervised setting. The `try` / `except` structure is a special form of control flow that can bypass run-time errors and allow a program to continue excuting. Optionally, custom error reporting can be added. This can also be useful for debugging programs.  

In [35]:
# divide by zero halts execution
values = list(range(-3, 4))
for i in values:
    print(f"input value {i} returns {42/i}")

input value -3 returns -14.0
input value -2 returns -21.0
input value -1 returns -42.0


ZeroDivisionError: division by zero

In [36]:
# handle a divide by zero error
values = list(range(-3, 4))
for i in values:
    try: 
        print(f"input value {i} returns {42/i}")
    except:
        print("Warning: divide by zero error!") 

input value -3 returns -14.0
input value -2 returns -21.0
input value -1 returns -42.0
input value 1 returns 42.0
input value 2 returns 21.0
input value 3 returns 14.0


In [58]:
# trapping specific kinds of errors
values = [-3, -1, 0, 1, 'a', 3]
for i in values:
    try: 
        print(f"Result: {42/i}")
    except ZeroDivisionError as e:
        print(f"Error: {e}")
    except TypeError as t:
        print(f"Error: {t}")

Result: -14.0
Result: -42.0
Error: division by zero
Result: 42.0
Error: unsupported operand type(s) for /: 'int' and 'str'
Result: 14.0


Python provides specific information about the type of error that caused an exception, which is very helpful for debugging code. It can sometimes be useful to create **custom exceptions**. For example, you may want to trap obviously incorrect temperature measurements by limiting allowable values to a specific range or a specific data type. To report a custom exception, use the `raise` keyword.

In [61]:
# trap obviously incorrect temperature values
values = [22, 24, 19, 3023, 26] 
for t in values:
    if (t < -20) or (t > 110): 
        raise ValueError(f"Temperature unrealistic: {t}")
    else:
        print(f"Temperature = {t}")

Temperature = 22
Temperature = 24
Temperature = 19


ValueError: Temperature unrealistic: 3023

To trap different kinds of errors and provide specific information about each, use one or more `elif` clauses.

In [64]:
# trap obviously incorrect temperature values and non-integer values
values = [22, 24, 19, 33.3, 26] 
for t in values:
    if (t < -20) or (t > 110): 
        raise ValueError(f"Temperature unrealistic: {t}")
    elif type(t) == float:
        raise TypeError(f"Value must be an integer: {t}")        
    else:
        print(f"Temperature = {t}")

Temperature = 22
Temperature = 24
Temperature = 19


ValueError: Temperature unrealistic: 3303